In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [15]:

NOTES = [
    "LangGraph gives an agent memory using a checkpointer, and a thread_id names one conversation. "
    "Same thread_id remembers earlier turns; a new thread_id starts fresh.",

    "A tool is an ordinary Python function with a docstring. The model reads the name, docstring "
    "and typed arguments to decide when and how to call it.",

    "create_agent(model, tools) builds the whole ReAct loop for you: it calls the model, runs the "
    "tool it asks for, feeds the result back, and repeats until done.",

    "RAG (retrieval-augmented generation) means: retrieve relevant text first, then let the model "
    "answer using that text, so answers are grounded in your documents instead of guessed.",

    "Saarathi Academy runs a 12-week AI Engineering and Machine Learning course, two hours a day, "
    "in Old Baneshwor, Kathmandu.",

    "The safe way to run arithmetic from a model is a small ast-based evaluator, never Python's "
    "eval(), because a tool is a door into your system.",
]


In [18]:


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key="AQ.Ab8RN6J2L4Ex8G2CpxgDjMyhQWzBjAdUodmxiU1aRIEVo5CxqQ",  
)


In [19]:
me = {"configurable": {"thread_id": "aug_26"}} 

In [24]:
_vec = TfidfVectorizer().fit(NOTES)          # build the index once
_M   = _vec.transform(NOTES)



@tool
def search_notes(query: str) -> str:
    """Search the user's course notes and return the most relevant note."""
    sims = cosine_similarity(_vec.transform([query]), _M)[0]
    return NOTES[int(sims.argmax())]  

@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression, e.g. '9 * 10'."""
    return safe_eval(expression)      # never eval() model input



In [23]:
TOOLS = [search_notes]

assistant = create_agent(model, tools=TOOLS, checkpointer=InMemorySaver())
me = {"configurable": {"thread_id": "another_thread_test"}} 

def say(config, text):
    r = assistant.invoke({"messages": [{"role": "user", "content": text}]}, config)
    print("agent:", r["messages"][-1].content)

    
#say(me, "save this fact 'ram is a good boy'")
say(me, "what is retrieval augmented generation?")



agent: [{'type': 'text', 'text': 'Based on your course notes, **RAG (retrieval-augmented generation)** is a technique where you **retrieve relevant text first, then let the model answer using that text**. This ensures the model\'s answers are grounded in your specific documents instead of guessed.\n\nHere is a simple breakdown of how it works:\n\n1. **Retrieve (Find):** When you ask a question, the system searches a database of your documents (like PDFs, articles, or notes) to find the text most relevant to your query.\n2. **Augment (Combine):** The system takes your original question and "augments" (combines) it with the relevant text snippets it just found.\n3. **Generate (Answer):** This combined prompt is sent to the Language Model (LLM). The model reads the provided text and generates an accurate, fact-based answer.\n\n### Why is RAG useful?\n* **Reduces Hallucinations:** Because the model has the exact source text in front of it, it is much less likely to make things up.\n* **Up-

In [26]:
TOOLS = [search_notes, calculator]           # a real choice to make
agent = create_agent(model, tools=TOOLS)

say(me, "How does agent memory work in LangGraph?")

agent: [{'type': 'text', 'text': 'In **LangGraph**, agent memory is managed using two core concepts: **checkpointers** and **thread IDs**. \n\nAccording to your course notes, here is how they work together:\n\n* **The Checkpointer:** This is the underlying mechanism that actually saves the agent\'s state (its memory) after each step in the graph. It writes the state to a database or memory cache.\n* **The Thread ID (`thread_id`):** This is a unique identifier that names a specific conversation. \n\n### How it operates in practice:\n* **Continuing a conversation (Same `thread_id`):** If you pass the same `thread_id` in your configuration details when calling the agent, LangGraph will look up the saved checkpoint for that thread. The agent will "remember" all the earlier turns and context from that conversation.\n* **Starting fresh (New `thread_id`):** If you pass a brand-new `thread_id` (or don\'t provide one), the agent starts with a blank slate, with no memory of past interactions.\n\